# Logistic Regression — Exercises

Companion to the note [Logistic Regression](Logistic%20Regression.md).

Practise the sigmoid, log-odds, binary cross-entropy and its gradient/Hessian, softmax, gradient descent and Newton/IRLS fitting, regularisation on separable data and calibration, first by hand and then from scratch in NumPy.

**18 exercises** · 🧠 Concept ×4 · ✍️ By hand ×8 · 💻 Code ×6

**How to use this notebook**
- Difficulty: ★☆☆ warm-up · ★★☆ standard · ★★★ challenge.
- ✍️ *By hand* exercises are meant for pen and paper (or a markdown cell). 💻 *Code* exercises have a cell for you to fill in.
- Every exercise has a folded **💡 Hint** and **✅ Solution**. Click to unfold, but try first!
- The notebook runs top to bottom as-is; `check(...)` lines print ⏳ until you have an answer, then ✅ or ❌.


In [ ]:
import numpy as np
np.random.seed(0)

# --- helper: compares your answer with a reference, prints ✅/❌, never raises ---
import numpy as _np
def check(name, got, expected, tol=1e-6):
    if got is None or got is Ellipsis:
        print(f"⏳ {name}: not attempted yet")
        return
    try:
        ok = bool(_np.allclose(_np.asarray(got, dtype=float), _np.asarray(expected, dtype=float), atol=tol, rtol=1e-4))
    except Exception:
        try:
            ok = bool(got == expected)
        except Exception:
            ok = False
    print(("✅ " if ok else "❌ ") + name + ("" if ok else f"  (got {got!r})"))

from scipy.special import expit, softmax as sp_softmax
from sklearn.linear_model import LogisticRegression
from sklearn.datasets import load_iris, load_breast_cancer
from sklearn.preprocessing import StandardScaler
rng = np.random.default_rng(0)
# Shared 2-D dataset with overlapping classes
X2 = rng.normal(size=(200, 2))
y2 = (X2 @ np.array([2.0, -1.0]) + 0.5 + rng.normal(size=200) > 0).astype(float)

## Part A · Concepts

### Exercise 1 · Why not linear regression on 0/1 labels?
*🧠 Concept · ★☆☆*

Give two reasons why fitting ordinary least squares to labels $y\in\{0,1\}$ and thresholding at $0.5$ is a poor classifier,
and explain how $p(y=1\mid x)=\sigma(w^\top x+b)$ fixes them.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Think about the range of $w^\top x+b$, and about what a far-away but *correctly* classified point does to a squared loss.

</details>

<details>
<summary><b>✅ Solution</b></summary>

1. **Outputs are not probabilities**: $w^\top x+b$ ranges over $\mathbb R$, so predictions $<0$ or $>1$ are meaningless. The sigmoid squashes to $(0,1)$.
2. **Squared loss punishes being 'too right'**: a point with $y=1$ and $\hat y=5$ has loss $16$, so outliers far on the correct side drag the boundary (see the least-squares row in [[Linear Models for Classification]]).
   Cross-entropy with a sigmoid gives such points near-zero loss.
3. (Bonus) The Gaussian-noise assumption behind least squares is wrong for binary targets; the Bernoulli likelihood is the right model.

</details>

### Exercise 2 · Separable data and diverging weights
*🧠 Concept · ★★☆*

The note says: *perfectly separable data ⇒ weights diverge*. Explain why the unregularised negative log-likelihood has no finite minimiser
in that case, and what adding $\frac\lambda2\lVert w\rVert^2$ changes.

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

If $w$ separates the data, consider $c\,w$ with $c\to\infty$. What happens to every $\hat p_i$ and to the loss?

</details>

<details>
<summary><b>✅ Solution</b></summary>

If $(w,b)$ separates the data, then $y_i=1 \Rightarrow w^\top x_i+b>0$ and $y_i=0 \Rightarrow <0$. Scaling to $(cw,cb)$ pushes every
$\hat p_i$ towards its label, so the loss decreases strictly towards $0$ as $c\to\infty$, but never reaches it: the infimum is not attained,
gradient descent keeps growing $\lVert w\rVert$, and the probabilities become over-confident (0 or 1).
With L2 the objective $L(w)+\frac\lambda2\lVert w\rVert^2$ is **strictly convex and coercive**, so a unique finite minimiser exists
([[Overfitting and Regularization]]).

</details>

### Exercise 3 · Discriminative vs generative, calibration
*🧠 Concept · ★☆☆*

(a) Why is logistic regression called *discriminative* while [[Naive Bayes]] is *generative*?
(b) The note says logistic regression gives *calibrated probabilities out of the box (unlike [[Support Vector Machines]])*. What does 'calibrated' mean,
and why does the log-loss encourage it?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

(a) Which distribution does each model write down: $p(y\mid x)$ or $p(x, y)$? (b) Log-loss is a *proper scoring rule*.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) Logistic regression models $p(y\mid x)$ directly. Naive Bayes models $p(x\mid y)p(y)$ and gets $p(y\mid x)$ by Bayes' rule.
With the same features, Gaussian NB with shared variances gives a posterior of exactly the logistic form, but the parameters are fitted differently.
(b) Calibrated: among all points predicted with probability $0.8$, about 80% are positive. The expected log-loss is minimised by predicting the true
conditional probability (a proper scoring rule), so the MLE is pushed towards calibrated outputs. An SVM minimises hinge loss, which only cares about the sign/margin,
so its scores need Platt scaling to become probabilities.

</details>

### Exercise 4 · Logistic regression as a neuron
*🧠 Concept · ★☆☆*

Draw logistic regression as a single neuron of a [[Neural Networks|neural network]]: what are the inputs, weights, activation, and loss?
Why does stacking two such layers *without* a nonlinearity in between give nothing new?

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Compose two affine maps $W_2(W_1x+b_1)+b_2$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Inputs $x_j$, weights $w_j$, bias $b$, pre-activation $z=w^\top x+b$, activation $\sigma(z)$, loss binary cross-entropy.
Two affine layers compose into one affine map $W_2W_1x+(W_2b_1+b_2)$, so the decision boundary stays a hyperplane. A hidden nonlinearity is what lets an MLP solve XOR.

</details>

## Part B · By hand

### Exercise 5 · Sigmoid identities
*✍️ By hand · ★☆☆*

Show that (i) $\sigma(-z)=1-\sigma(z)$ and (ii) $\sigma'(z)=\sigma(z)\,(1-\sigma(z))$. Then show that the inverse of $\sigma$ is the **logit**
$z=\log\frac{p}{1-p}$ (log-odds).

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Multiply numerator and denominator of $\frac{1}{1+e^{z}}$ by $e^{-z}$. For (ii), differentiate $(1+e^{-z})^{-1}$ with the chain rule.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(i) $\sigma(-z)=\frac1{1+e^{z}}=\frac{e^{-z}}{e^{-z}+1}=1-\frac{1}{1+e^{-z}}=1-\sigma(z)$.
(ii) $\sigma'(z)=\frac{e^{-z}}{(1+e^{-z})^2}=\frac{1}{1+e^{-z}}\cdot\frac{e^{-z}}{1+e^{-z}}=\sigma(z)(1-\sigma(z))$.
Logit: $p=\frac1{1+e^{-z}} \Rightarrow e^{-z}=\frac{1-p}{p} \Rightarrow z=\log\frac{p}{1-p}$.

</details>

### Exercise 6 · Reading a coefficient
*✍️ By hand · ★☆☆*

A model predicting loan default has coefficient $w_{\text{debt}}=0.7$ for 'debt in units of 10k€'. By what **factor** do the odds of default
change when debt increases by 10k€ (all else fixed)? Store it in `odds_factor`. Does the *probability* change by a fixed amount too?

In [ ]:
odds_factor = None

check('odds factor', odds_factor, np.exp(0.7), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\log\text{odds}=w^\top x+b$, so a unit step in $x_j$ adds $w_j$ to the log-odds.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Odds are multiplied by $e^{0.7}\approx 2.014$ (roughly doubled). The probability change is **not** constant: it is largest near $p=0.5$
(slope $\sigma'(z)=p(1-p)\le 1/4$) and tiny when $p$ is near 0 or 1.

```python
odds_factor = np.exp(0.7)
```

</details>

### Exercise 7 · A forward pass
*✍️ By hand · ★☆☆*

With $w=(2,-1)^\top$, $b=0.5$ and $x=(1,3)^\top$, compute $z=w^\top x+b$ and $\hat p=\sigma(z)$. Which class is predicted with threshold $0.5$?

In [ ]:
z_fp = None
p_fp = None

check('z', z_fp, np.array([2,-1])@np.array([1,3])+0.5)
check('p', p_fp, expit(np.array([2,-1])@np.array([1,3])+0.5), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$\sigma(-0.5) = 1/(1+e^{0.5})$ and $e^{0.5}\approx 1.6487$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$z=2-3+0.5=-0.5$, $\hat p=\sigma(-0.5)=1/(1+1.6487)\approx 0.3775<0.5$, so class 0 is predicted.

```python
z_fp = -0.5
p_fp = 1 / (1 + np.exp(0.5))
```

</details>

### Exercise 8 · Binary cross-entropy by hand
*✍️ By hand · ★☆☆*

Three points have labels $y=(1,0,1)$ and predictions $\hat p=(0.9, 0.2, 0.6)$. Compute the loss $L=-\sum_i y_i\log\hat p_i+(1-y_i)\log(1-\hat p_i)$
(natural log). Which point contributes most?

In [ ]:
bce_hand = None

_y=np.array([1,0,1.]); _p=np.array([.9,.2,.6])
check('BCE', bce_hand, -np.sum(_y*np.log(_p)+(1-_y)*np.log(1-_p)), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Only one of the two terms is active per point: $-\log 0.9$, $-\log 0.8$, $-\log 0.6$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$L=-\log0.9-\log0.8-\log0.6=0.1054+0.2231+0.5108\approx 0.8393$. The third point (a positive predicted with only $0.6$) contributes most.

```python
bce_hand = -(np.log(0.9) + np.log(0.8) + np.log(0.6))
```

</details>

### Exercise 9 · Deriving the gradient
*✍️ By hand · ★★☆*

Show that the gradient of the binary cross-entropy is $\nabla_w L=X^\top(\hat p-y)$, where $\hat p_i=\sigma(w^\top x_i)$ (bias absorbed into $x$).
Compare with the least-squares gradient of [[Linear Regression]].

*Your answer:* ✏️ (double-click to write here)

<details>
<summary><b>💡 Hint</b></summary>

Per point, $\frac{\partial L_i}{\partial z_i}$ simplifies beautifully. Use $\sigma'=\sigma(1-\sigma)$ and the chain rule $\partial z_i/\partial w = x_i$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$L_i=-y_i\log\sigma(z_i)-(1-y_i)\log(1-\sigma(z_i))$.
$\frac{\partial L_i}{\partial z_i}=-y_i\frac{\sigma(1-\sigma)}{\sigma}+(1-y_i)\frac{\sigma(1-\sigma)}{1-\sigma}=-y_i(1-\sigma)+(1-y_i)\sigma=\sigma(z_i)-y_i$.
Hence $\nabla_w L=\sum_i(\hat p_i-y_i)x_i=X^\top(\hat p-y)$. It has the same 'error times input' form as least squares, $X^\top(Xw-y)$,
but the prediction passes through $\sigma$, so setting it to zero is non-linear in $w$: no closed form.

</details>

### Exercise 10 · Hessian and convexity
*✍️ By hand · ★★☆*

(a) Show that the Hessian is $H=X^\top S X$ with $S=\operatorname{diag}(\hat p_i(1-\hat p_i))$, and conclude that $L$ is convex.
(b) For $X=\begin{pmatrix}1&0\\1&1\\1&2\end{pmatrix}$ (first column = bias) and $w=0$, compute $H$ and store it in `H_hand`.

In [ ]:
H_hand = None  # 2x2 list or array

_X=np.array([[1,0],[1,1],[1,2.]]); _p=expit(_X@np.zeros(2))
check('Hessian', H_hand, _X.T@np.diag(_p*(1-_p))@_X)

<details>
<summary><b>💡 Hint 1</b></summary>

Differentiate $\sum_i(\sigma(w^\top x_i)-y_i)x_i$ once more with respect to $w^\top$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

At $w=0$ every $\hat p_i=1/2$, so $S=\tfrac14 I$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $\nabla^2 L=\sum_i \sigma'(z_i)x_ix_i^\top=X^\top S X$. For any $v$: $v^\top Hv=\sum_i s_i (x_i^\top v)^2\ge0$ since $s_i\in(0,\tfrac14]$, so $H$ is PSD and $L$ is convex.
(b) $H=\tfrac14 X^\top X=\tfrac14\begin{pmatrix}3&3\\3&5\end{pmatrix}=\begin{pmatrix}0.75&0.75\\0.75&1.25\end{pmatrix}$.

```python
H_hand = [[0.75, 0.75], [0.75, 1.25]]
```

</details>

### Exercise 11 · Softmax and its gradient
*✍️ By hand · ★★☆*

For logits $z=(2,1,0)$ compute the softmax probabilities $p_k=e^{z_k}/\sum_je^{z_j}$, the cross-entropy loss when the true class is the
**second** one ($y=(0,1,0)$), and the gradient $\partial L/\partial z=p-y$.

In [ ]:
p_soft = None     # 3 probabilities
ce_soft = None
grad_soft = None  # 3 numbers

_p=sp_softmax([2.,1.,0.])
check('softmax', p_soft, _p, tol=1e-3)
check('cross-entropy', ce_soft, -np.log(_p[1]), tol=1e-3)
check('gradient', grad_soft, _p-np.array([0,1,0]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

$e^2\approx7.389$, $e^1\approx2.718$, $e^0=1$, sum $\approx 11.107$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

$p\approx(0.6652,\ 0.2447,\ 0.0900)$. $L=-\log 0.2447\approx1.4076$. Gradient $p-y\approx(0.6652,\ -0.7553,\ 0.0900)$:
increase the logit of the true class, decrease the others in proportion to their probability. With $K=2$ softmax reduces to the sigmoid of $z_1-z_0$.

```python
p_soft = np.exp([2, 1, 0]) / np.exp([2, 1, 0]).sum()
ce_soft = -np.log(p_soft[1])
grad_soft = p_soft - np.array([0, 1, 0])
```

</details>

### Exercise 12 · Geometry of the decision boundary
*✍️ By hand · ★☆☆*

A 2-D model has $w=(1,2)^\top$, $b=-4$. (a) Where does the boundary $w^\top x+b=0$ cross the axes? (b) What is $p(y=1\mid x)$ at $x=(3,1)$?
(c) What is the (perpendicular) distance from the origin to the boundary?

In [ ]:
x1_cross = None  # where the boundary meets the x1-axis
x2_cross = None  # where it meets the x2-axis
p_31 = None
dist0 = None

check('x1 intercept', x1_cross, 4)
check('x2 intercept', x2_cross, 2)
check('p(3,1)', p_31, expit(3+2-4), tol=1e-3)
check('distance', dist0, 4/np.linalg.norm([1,2]), tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Distance from a point $x_0$ to the hyperplane is $|w^\top x_0+b|/\lVert w\rVert$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

(a) $x_1+2x_2=4$: crosses at $x_1=4$ and $x_2=2$. (b) $z=3+2-4=1$, $p=\sigma(1)\approx0.731$. (c) $|b|/\lVert w\rVert=4/\sqrt5\approx1.789$.
Note $w$ is normal to the boundary, and $|z|/\lVert w\rVert$ is a signed distance: the confidence grows with distance from the boundary.

```python
x1_cross = 4
x2_cross = 2
p_31 = 1 / (1 + np.exp(-1))
dist0 = 4 / np.sqrt(5)
```

</details>

## Part C · Code from scratch

### Exercise 13 · A numerically stable sigmoid and log-loss
*💻 Code · ★☆☆*

`1/(1+np.exp(-z))` overflows for $z=-1000$ (warning), and `np.log(sigmoid(z))` gives `-inf` for large negative $z$.
Implement `sigmoid(z)` that never overflows, and `log_sigmoid(z)` $=\log\sigma(z)$ computed stably.

In [ ]:
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    # TODO
    return None

def log_sigmoid(z):
    z = np.asarray(z, dtype=float)
    # TODO: log sigma(z) = -log(1 + e^{-z}); use np.logaddexp
    return None

z_test = np.array([-1000., -30., -1., 0., 2., 40., 1000.])

check('sigmoid', sigmoid(z_test), expit(z_test))
check('log_sigmoid', log_sigmoid(z_test), -np.logaddexp(0, -z_test))

<details>
<summary><b>💡 Hint 1</b></summary>

Split cases: for $z\ge0$ use $1/(1+e^{-z})$; for $z<0$ use $e^{z}/(1+e^{z})$. Then the exponent is always $\le 0$.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

`np.logaddexp(a, b)` computes $\log(e^a+e^b)$ safely, so $\log\sigma(z)=-$`np.logaddexp(0, -z)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The trick is to only ever exponentiate non-positive numbers. In the log domain $\log\sigma(z)=-\log(1+e^{-z})=-\operatorname{softplus}(-z)$.
The BCE can then be written as $L_i = \operatorname{softplus}(z_i) - y_i z_i$, which is how libraries compute it from logits.

```python
def sigmoid(z):
    z = np.asarray(z, dtype=float)
    out = np.empty_like(z)
    pos = z >= 0
    out[pos] = 1 / (1 + np.exp(-z[pos]))
    ez = np.exp(z[~pos])
    out[~pos] = ez / (1 + ez)
    return out

def log_sigmoid(z):
    z = np.asarray(z, dtype=float)
    return -np.logaddexp(0, -z)

z_test = np.array([-1000., -30., -1., 0., 2., 40., 1000.])
```

</details>

### Exercise 14 · Logistic regression with gradient descent
*💻 Code · ★★☆*

Implement `fit_logreg_gd(X, y, lr, epochs)` (adds a bias column, minimises the **mean** cross-entropy with batch gradient descent, returns
$w$ with bias first) and `bce(X, y, w)` (mean loss). Fit it on the shared data `X2, y2`. Your loss should match the unregularised
sklearn fit (`C=np.inf`) to 3 decimals.

In [ ]:
def bce(X, y, w):
    # TODO: mean binary cross-entropy, X WITHOUT bias column, w = [b, w1, ...]
    return None

def fit_logreg_gd(X, y, lr=0.5, epochs=3000):
    # TODO
    return None

w_gd = fit_logreg_gd(X2, y2)

_m = LogisticRegression(C=np.inf, tol=1e-10, max_iter=5000).fit(X2, y2)
_wref = np.r_[_m.intercept_, _m.coef_.ravel()]
_Xb = np.c_[np.ones(len(X2)), X2]; _pr = expit(_Xb @ _wref)
_lref = -np.mean(y2*np.log(_pr) + (1-y2)*np.log(1-_pr))
check('bce at sklearn solution', bce(X2, y2, _wref), _lref)
check('GD reaches the MLE loss', None if w_gd is None else bce(X2, y2, w_gd), _lref, tol=1e-3)

<details>
<summary><b>💡 Hint</b></summary>

Gradient of the mean loss: $\frac1N X_b^\top(\sigma(X_bw)-y)$. Initialise $w=0$.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Because the loss is convex, GD with a small enough step converges to the global minimum. The Hessian of the mean loss is bounded by
$\frac{1}{4N}X^\top X$, so any $\eta<8/\lambda_{\max}(X^\top X/N)$ is safe ([[Gradient Descent]]).

```python
def bce(X, y, w):
    z = np.c_[np.ones(len(X)), X] @ w
    return np.mean(np.logaddexp(0, z) - y * z)

def fit_logreg_gd(X, y, lr=0.5, epochs=3000):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(epochs):
        w -= lr * Xb.T @ (expit(Xb @ w) - y) / len(y)
    return w

w_gd = fit_logreg_gd(X2, y2)
print('GD weights', w_gd)
```

</details>

### Exercise 15 · Newton's method / IRLS
*💻 Code · ★★☆*

Implement `fit_irls(X, y, iters)` using Newton steps $w\leftarrow w-H^{-1}\nabla L$ with $\nabla L=X^\top(\hat p-y)$ and $H=X^\top SX$.
Fit it on `X2, y2` and compare the coefficients with sklearn. How many iterations do you need compared to GD?

In [ ]:
def fit_irls(X, y, iters=10):
    # TODO: bias column, w = 0, Newton updates; return [b, w1, ...]
    return None

w_irls = fit_irls(X2, y2)

_m = LogisticRegression(C=np.inf, tol=1e-10, max_iter=5000).fit(X2, y2)
check('IRLS = sklearn', w_irls, np.r_[_m.intercept_, _m.coef_.ravel()], tol=1e-4)

<details>
<summary><b>💡 Hint</b></summary>

`H = Xb.T @ (Xb * s[:, None])` with `s = p*(1-p)`; then `w -= np.linalg.solve(H, g)`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Newton converges quadratically: 6–8 iterations reach machine precision, versus thousands of GD steps. Each step solves a $d\times d$ system,
which is cheap for small $d$. The name IRLS comes from rewriting the Newton step as a *weighted least-squares* problem with weights $s_i$
and working response $z_i + (y_i-\hat p_i)/s_i$.

```python
def fit_irls(X, y, iters=10):
    Xb = np.c_[np.ones(len(X)), X]
    w = np.zeros(Xb.shape[1])
    for _ in range(iters):
        p = expit(Xb @ w)
        g = Xb.T @ (p - y)
        H = Xb.T @ (Xb * (p * (1 - p))[:, None])
        w -= np.linalg.solve(H, g)
    return w

w_irls = fit_irls(X2, y2)
print('IRLS weights', w_irls)
```

</details>

### Exercise 16 · Watch the weights diverge
*💻 Code · ★★☆*

Create a perfectly separable 1-D dataset `x_sep = [-2,-1,-0.5,0.5,1,2]`, `y_sep = [0,0,0,1,1,1]`. Run gradient descent on the mean cross-entropy
for 20 000 steps (lr 0.5), once without and once with an L2 penalty $\frac\lambda2 w^2$, $\lambda=0.1$ (on the slope only).
Store the final slope magnitudes in `w_noreg` and `w_l2`.

In [ ]:
x_sep = np.array([-2, -1, -0.5, 0.5, 1, 2.])
y_sep = np.array([0, 0, 0, 1, 1, 1.])
w_noreg = None
w_l2 = None

check('unregularised slope keeps growing (> 8)', None if w_noreg is None else w_noreg > 8, True)
_m = LogisticRegression(C=1/(0.1*len(x_sep)), tol=1e-10).fit(x_sep[:, None], y_sep)
check('L2 slope matches sklearn', w_l2, abs(_m.coef_[0, 0]), tol=1e-3)

<details>
<summary><b>💡 Hint 1</b></summary>

Keep $(b,w)$; the gradient of the mean loss is $\frac1N\sum(\hat p_i-y_i)(1,x_i)$, plus $(0,\lambda w)$ for the penalty.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

sklearn minimises $C\sum_i L_i+\frac12w^2$; dividing by $CN$ shows $C=1/(\lambda N)$ is the same problem.

</details>

<details>
<summary><b>✅ Solution</b></summary>

Without regularisation the slope grows like $\log t$ forever (it is ≈ 13.5 after 20 000 steps and still rising), the predicted probabilities approach 0/1.
With $\lambda=0.1$ GD converges to a finite slope ($\approx 1.48$) and the probabilities stay moderate.

```python
x_sep = np.array([-2, -1, -0.5, 0.5, 1, 2.])
y_sep = np.array([0, 0, 0, 1, 1, 1.])

def gd_1d(lam, steps=20000, lr=0.5):
    b = w = 0.0
    for _ in range(steps):
        r = expit(b + w * x_sep) - y_sep
        b -= lr * r.mean()
        w -= lr * (np.mean(r * x_sep) + lam * w)
    return w

w_noreg = abs(gd_1d(0.0))
w_l2 = abs(gd_1d(0.1))
print(w_noreg, w_l2)
```

</details>

### Exercise 17 · Softmax regression on Iris
*💻 Code · ★★★*

Implement multinomial logistic regression: `softmax(Z)` (row-wise, stable), `ce_grad(W, Xb, Y)` returning the mean cross-entropy and its gradient
$\frac1N X_b^\top(P-Y)$ for one-hot $Y$, and train with GD on standardised Iris. (a) Verify your gradient with finite differences.
(b) Reach training accuracy ≥ 0.95.

In [ ]:
def softmax(Z):
    # TODO: subtract the row max before exponentiating
    return None

def ce_grad(W, Xb, Y):
    # TODO: return (loss, grad) with W of shape (d+1, K)
    return None, None

Xi, yi = load_iris(return_X_y=True)
Xi = StandardScaler().fit_transform(Xi)
Xbi = np.c_[np.ones(len(Xi)), Xi]
Yi = np.eye(3)[yi]
W_soft = None   # trained weights
acc_soft = None

_W = np.random.default_rng(1).normal(size=(5, 3))
_l, _g = ce_grad(_W, Xbi, Yi)
if _g is None:
    check('gradient vs finite differences', None, 0)
else:
    _num = np.zeros_like(_W); _e = 1e-6
    for _i in range(5):
        for _k in range(3):
            _E = np.zeros_like(_W); _E[_i, _k] = _e
            _num[_i, _k] = (ce_grad(_W + _E, Xbi, Yi)[0] - ce_grad(_W - _E, Xbi, Yi)[0]) / (2*_e)
    check('gradient vs finite differences', _g, _num, tol=1e-6)
check('softmax rows', None if softmax(np.array([[1000., 0.]])) is None else softmax(np.array([[1000., 0.], [0, 0]])), sp_softmax(np.array([[1000., 0.], [0, 0]]), axis=1))
check('training accuracy >= 0.95', None if acc_soft is None else acc_soft >= 0.95, True)

<details>
<summary><b>💡 Hint 1</b></summary>

`Z - Z.max(axis=1, keepdims=True)` does not change the softmax but prevents overflow.

</details>

<details>
<summary><b>💡 Hint 2</b></summary>

Loss $=-\frac1N\sum_i\sum_k Y_{ik}\log P_{ik}$; train with e.g. `lr=0.5` for 2000 steps from `W=0`.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The gradient matches the note's '$\hat p-y$' rule in matrix form. Iris is nearly separable, so accuracy reaches about 0.97–0.98; without regularisation
the weights would keep growing slowly on the separable setosa class (Exercise 16).

```python
def softmax(Z):
    Z = Z - Z.max(axis=1, keepdims=True)
    E = np.exp(Z)
    return E / E.sum(axis=1, keepdims=True)

def ce_grad(W, Xb, Y):
    P = softmax(Xb @ W)
    loss = -np.mean(np.sum(Y * np.log(P + 1e-300), axis=1))
    return loss, Xb.T @ (P - Y) / len(Xb)

Xi, yi = load_iris(return_X_y=True)
Xi = StandardScaler().fit_transform(Xi)
Xbi = np.c_[np.ones(len(Xi)), Xi]
Yi = np.eye(3)[yi]
W_soft = np.zeros((5, 3))
for _ in range(2000):
    W_soft -= 0.5 * ce_grad(W_soft, Xbi, Yi)[1]
acc_soft = np.mean((Xbi @ W_soft).argmax(1) == yi)
print('training accuracy', acc_soft)
```

</details>

### Exercise 18 · A calibration check
*💻 Code · ★★★*

On standardised breast-cancer data (`load_breast_cancer`), split 50/50 with a fixed permutation, fit `LogisticRegression()` on the first half and
compute on the second half a **reliability table**: bin predicted probabilities into 5 equal-width bins and compare the mean prediction with the
observed positive rate per bin. Store the **expected calibration error**
$\text{ECE}=\sum_b \frac{n_b}{N}\,|\bar p_b-\bar y_b|$ in `ece`.

In [ ]:
Xc, yc = load_breast_cancer(return_X_y=True)
perm = np.random.default_rng(0).permutation(len(yc))
tr, te = perm[:len(yc)//2], perm[len(yc)//2:]
ece = None

from sklearn.calibration import calibration_curve
_sc = StandardScaler().fit(Xc[tr]); _m = LogisticRegression().fit(_sc.transform(Xc[tr]), yc[tr])
_p = _m.predict_proba(_sc.transform(Xc[te]))[:, 1]
_b = np.minimum((_p * 5).astype(int), 4)
_ece = sum(np.mean(_b == k) * abs(_p[_b == k].mean() - yc[te][_b == k].mean()) for k in range(5) if np.any(_b == k))
check('ECE', ece, _ece, tol=1e-6)
check('logistic regression is well calibrated (ECE < 0.1)', None if ece is None else ece < 0.1, True)

<details>
<summary><b>💡 Hint</b></summary>

Bin index: `np.minimum((p * 5).astype(int), 4)`. Skip empty bins. Fit the scaler on the training half only.

</details>

<details>
<summary><b>✅ Solution</b></summary>

The ECE is small (a few percent): most bins have mean prediction close to the observed rate, as expected from log-loss training.
Compare with an SVM's `decision_function` mapped to $[0,1]$, which is typically far worse calibrated. `sklearn.calibration.calibration_curve`
computes the same table.

```python
Xc, yc = load_breast_cancer(return_X_y=True)
perm = np.random.default_rng(0).permutation(len(yc))
tr, te = perm[:len(yc)//2], perm[len(yc)//2:]
sc = StandardScaler().fit(Xc[tr])
clf = LogisticRegression().fit(sc.transform(Xc[tr]), yc[tr])
p_te = clf.predict_proba(sc.transform(Xc[te]))[:, 1]
bins = np.minimum((p_te * 5).astype(int), 4)
ece = 0.0
for k in range(5):
    m = bins == k
    if m.any():
        print(f'bin {k}: n={m.sum():3d}  mean p={p_te[m].mean():.3f}  pos rate={yc[te][m].mean():.3f}')
        ece += m.mean() * abs(p_te[m].mean() - yc[te][m].mean())
print('ECE', ece)
```

</details>

---
*Done? Revisit the exercises you needed hints for in a few days — retrieval beats rereading. Back to [Logistic Regression](Logistic%20Regression.md).*